In [ ]:
"""
ELECTRICITY DEMAND FORECASTING & PEAK RISK DETECTION SYSTEM
============================================================
Optimized for best accuracy with PKL file outputs
"""

# ============================================================================
# 1. IMPORTS AND CONFIGURATION
# ============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Machine Learning
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                           r2_score, classification_report, confusion_matrix,
                           f1_score, precision_score, recall_score, roc_auc_score)
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingRegressor

# XGBoost
import xgboost as xgb

# LightGBM (for better performance)
import lightgbm as lgb

# System
import joblib
import os
import pickle
from typing import Tuple, Dict, List, Any
import logging
from scipy import stats
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('demand_forecast.log'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# Set random seeds for reproducibility
np.random.seed(42)

# ============================================================================
# 2. DATA LOADING AND PREPROCESSING
# ============================================================================

class DataLoaderPreprocessor:
    """
    Handles data loading, cleaning, and feature engineering
    """

    def __init__(self, filepath: str):
        self.filepath = filepath
        self.data = None
        self.feature_names = None

    def load_and_clean(self) -> pd.DataFrame:
        """
        Load CSV and perform initial cleaning
        """
        logger.info(f"Loading data from {self.filepath}")

        # Load data
        df = pd.read_csv(self.filepath)
        logger.info(f"Initial shape: {df.shape}")

        # Convert datetime
        df['datetime'] = pd.to_datetime(df['datetime'])

        # Sort chronologically
        df = df.sort_values('datetime').reset_index(drop=True)

        # Remove duplicate hour column
        if 'hour' in df.columns:
            df = df.drop(columns=['hour'])
            logger.info("Removed duplicate hour column")

        # Handle missing values
        missing = df.isnull().sum()
        if missing.any():
            logger.warning(f"Missing values found:\n{missing[missing > 0]}")
            # Interpolate for better time series handling
            df = df.interpolate(method='linear', limit_direction='both')

        # Remove low variance features
        self._remove_low_variance_features(df)

        # Stationarity check for target
        self._check_stationarity(df, 'load_demand_MW')

        self.data = df
        logger.info(f"Final shape after cleaning: {df.shape}")
        return df

    def _remove_low_variance_features(self, df: pd.DataFrame, threshold: float = 0.01):
        """
        Remove features with near-zero variance
        """
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        low_var_features = []

        for col in numeric_cols:
            if col == 'load_demand_MW':
                continue
            if df[col].std() < threshold * df[col].mean() if df[col].mean() != 0 else df[col].std() < 0.001:
                low_var_features.append(col)
                logger.info(f"Removing low variance feature: {col}")

        if low_var_features:
            df.drop(columns=low_var_features, inplace=True, errors='ignore')

    def _check_stationarity(self, df: pd.DataFrame, column: str):
        """
        Check stationarity of target variable
        """
        result = adfuller(df[column].dropna())
        logger.info(f'ADF Statistic for {column}: {result[0]:.4f}')
        logger.info(f'p-value: {result[1]:.4f}')
        if result[1] > 0.05:
            logger.warning(f"{column} is non-stationary. Consider differencing.")

    def create_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Create advanced time-based features
        """
        logger.info("Creating advanced time features...")

        # Basic time features
        df['year'] = df['datetime'].dt.year
        df['month'] = df['datetime'].dt.month
        df['week'] = df['datetime'].dt.isocalendar().week
        df['day'] = df['datetime'].dt.day
        df['day_of_week'] = df['datetime'].dt.dayofweek
        df['hour_of_day'] = df['datetime'].dt.hour
        df['quarter'] = df['datetime'].dt.quarter
        df['day_of_year'] = df['datetime'].dt.dayofyear

        # Cyclical encoding
        df['hour_sin'] = np.sin(2 * np.pi * df['hour_of_day'] / 24)
        df['hour_cos'] = np.cos(2 * np.pi * df['hour_of_day'] / 24)
        df['day_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
        df['day_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
        df['month_sin'] = np.sin(2 * np.pi * (df['month'] - 1) / 12)
        df['month_cos'] = np.cos(2 * np.pi * (df['month'] - 1) / 12)

        # Weekend and holiday indicators
        df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
        if 'is_holiday' not in df.columns:
            # Create holiday indicator based on weekends and common holidays
            df['is_holiday'] = df['is_weekend'].copy()

        # Time blocks
        df['time_block'] = pd.cut(df['hour_of_day'],
                                   bins=[-1, 6, 12, 18, 23],
                                   labels=['night', 'morning', 'afternoon', 'evening'])
        df = pd.get_dummies(df, columns=['time_block'], prefix='time')

        # Season (if not present)
        if 'season' not in df.columns:
            df['season'] = df['month'].apply(lambda x:
                'Winter' if x in [12, 1, 2] else
                'Spring' if x in [3, 4, 5] else
                'Summer' if x in [6, 7, 8] else 'Fall')

        return df

    def detect_outliers(self, df: pd.DataFrame, column: str = 'load_demand_MW') -> pd.DataFrame:
        """
        Detect and handle outliers using IQR and Z-score methods
        """
        # IQR method
        Q1 = df[column].quantile(0.25)
        Q3 = df[column].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        # Z-score method
        z_scores = np.abs(stats.zscore(df[column].dropna()))
        z_threshold = 3

        outliers_iqr = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
        outliers_z = df[z_scores > z_threshold] if len(z_scores) > 0 else pd.DataFrame()

        logger.info(f"IQR outliers: {len(outliers_iqr)} ({len(outliers_iqr)/len(df)*100:.2f}%)")
        logger.info(f"Z-score outliers: {len(outliers_z)} ({len(outliers_z)/len(df)*100:.2f}%)")

        # Cap extreme outliers (only beyond 3*IQR)
        extreme_lower = Q1 - 3 * IQR
        extreme_upper = Q3 + 3 * IQR
        df[column] = df[column].clip(extreme_lower, extreme_upper)

        return df


# ============================================================================
# 3. ADVANCED FEATURE ENGINEERING
# ============================================================================

class AdvancedFeatureEngineer:
    """
    Creates advanced features for optimal time series forecasting
    """

    def __init__(self, df: pd.DataFrame, target_col: str = 'load_demand_MW'):
        self.df = df.copy()
        self.target = target_col
        self.feature_importance = None

    def create_lag_features(self, lags: List[int] = None) -> pd.DataFrame:
        """
        Create optimal lag features based on autocorrelation
        """
        if lags is None:
            # Auto-select lags based on ACF
            lags = self._select_optimal_lags()

        logger.info(f"Creating lag features: {lags}")

        for lag in lags:
            self.df[f'lag_{lag}'] = self.df[self.target].shift(lag)

        return self.df

    def _select_optimal_lags(self, max_lag: int = 168) -> List[int]:
        """
        Select optimal lags based on autocorrelation
        """
        # Calculate autocorrelation
        acf_values = [1]  # lag 0
        series = self.df[self.target].values

        for lag in range(1, min(max_lag + 1, len(series) // 4)):
            corr = np.corrcoef(series[lag:], series[:-lag])[0, 1]
            acf_values.append(abs(corr) if not np.isnan(corr) else 0)

        # Select lags with significant autocorrelation (> 0.3)
        significant_lags = [i for i, val in enumerate(acf_values) if val > 0.3 and i > 0]

        # Ensure critical lags are included
        critical_lags = [1, 2, 3, 6, 12, 24, 48, 168]
        selected_lags = list(set(significant_lags + critical_lags))
        selected_lags.sort()

        logger.info(f"Selected lags based on ACF: {selected_lags}")
        return selected_lags[:15]  # Limit to 15 most important lags

    def create_rolling_features(self, windows: List[int] = None) -> pd.DataFrame:
        """
        Create rolling statistics with optimal windows
        """
        if windows is None:
            windows = [6, 12, 24, 48, 168]  # 6h, 12h, 1d, 2d, 1w

        logger.info(f"Creating rolling features with windows: {windows}")

        for window in windows:
            # Basic statistics
            self.df[f'roll_mean_{window}'] = self.df[self.target].rolling(window=window, min_periods=1).mean()
            self.df[f'roll_std_{window}'] = self.df[self.target].rolling(window=window, min_periods=1).std()
            self.df[f'roll_min_{window}'] = self.df[self.target].rolling(window=window, min_periods=1).min()
            self.df[f'roll_max_{window}'] = self.df[self.target].rolling(window=window, min_periods=1).max()
            self.df[f'roll_median_{window}'] = self.df[self.target].rolling(window=window, min_periods=1).median()

            # Rate of change
            self.df[f'roll_change_{window}'] = self.df[self.target].rolling(window=window, min_periods=1).apply(
                lambda x: x.iloc[-1] - x.iloc[0] if len(x) > 1 else 0, raw=False
            )

            # Coefficient of variation (normalized volatility)
            self.df[f'roll_cv_{window}'] = (self.df[f'roll_std_{window}'] /
                                            (self.df[f'roll_mean_{window}'] + 1e-8))

        return self.df

    def create_expanding_features(self) -> pd.DataFrame:
        """
        Create expanding window features (historical averages)
        """
        logger.info("Creating expanding window features...")

        self.df['expanding_mean'] = self.df[self.target].expanding(min_periods=24).mean()
        self.df['expanding_std'] = self.df[self.target].expanding(min_periods=24).std()
        self.df['expanding_min'] = self.df[self.target].expanding(min_periods=24).min()
        self.df['expanding_max'] = self.df[self.target].expanding(min_periods=24).max()

        return self.df

    def create_interaction_features(self) -> pd.DataFrame:
        """
        Create advanced interaction terms
        """
        logger.info("Creating interaction features...")

        # Temperature interactions
        if 'temperature_C' in self.df.columns:
            self.df['temp_hour'] = self.df['temperature_C'] * self.df['hour_sin']
            self.df['temp_weekend'] = self.df['temperature_C'] * self.df['is_weekend']
            self.df['temp_squared'] = self.df['temperature_C'] ** 2
            self.df['temp_cubed'] = self.df['temperature_C'] ** 3

            # Temperature change rate
            self.df['temp_change_1h'] = self.df['temperature_C'].diff(1)
            self.df['temp_change_3h'] = self.df['temperature_C'].diff(3)
            self.df['temp_change_24h'] = self.df['temperature_C'].diff(24)

        # Humidity interactions
        if 'humidity_%' in self.df.columns:
            self.df['humidity_hour'] = self.df['humidity_%'] * self.df['hour_cos']
            self.df['humidity_temp'] = self.df['humidity_%'] * self.df['temperature_C'] if 'temperature_C' in self.df.columns else 0

        # Solar interactions
        if 'solar_irradiation_Wm2' in self.df.columns:
            self.df['solar_hour'] = self.df['solar_irradiation_Wm2'] * self.df['hour_sin']
            self.df['solar_temp'] = self.df['solar_irradiation_Wm2'] * self.df['temperature_C'] if 'temperature_C' in self.df.columns else 0

        # Price interactions
        if 'power_price_usd_per_kwh' in self.df.columns:
            self.df['price_hour'] = self.df['power_price_usd_per_kwh'] * self.df['hour_sin']
            self.df['price_change_1h'] = self.df['power_price_usd_per_kwh'].diff(1)

        return self.df

    def create_peak_label(self, threshold_percentile: float = 90) -> pd.DataFrame:
        """
        Create binary target for peak demand classification
        """
        peak_threshold = self.df[self.target].quantile(threshold_percentile / 100)
        self.df['is_peak'] = (self.df[self.target] > peak_threshold).astype(int)

        logger.info(f"Peak threshold ({threshold_percentile}th percentile): {peak_threshold:.2f} MW")
        logger.info(f"Peak hours: {self.df['is_peak'].sum()} ({(self.df['is_peak'].mean()*100):.2f}%)")

        return self.df

    def remove_nan_rows(self) -> pd.DataFrame:
        """
        Remove rows with NaN values
        """
        initial_rows = len(self.df)
        self.df = self.df.dropna()
        removed = initial_rows - len(self.df)
        logger.info(f"Removed {removed} rows with NaN values")
        return self.df

    def get_feature_correlation(self) -> pd.DataFrame:
        """
        Get correlation matrix for feature selection
        """
        numeric_cols = self.df.select_dtypes(include=[np.number]).columns
        corr_matrix = self.df[numeric_cols].corr()

        # Get correlation with target
        target_corr = corr_matrix[self.target].abs().sort_values(ascending=False)
        logger.info(f"Top 10 features correlated with {self.target}:\n{target_corr.head(11)}")

        return corr_matrix


# ============================================================================
# 4. PEAK DEMAND ANALYSIS
# ============================================================================

class PeakAnalyzer:
    """
    Analyzes peak demand patterns and characteristics
    """

    def __init__(self, df: pd.DataFrame):
        self.df = df
        self.peak_df = None
        self.analysis_results = None

    def analyze_peak_patterns(self) -> Dict:
        """
        Comprehensive peak analysis
        """
        logger.info("Analyzing peak demand patterns...")

        if 'is_peak' not in self.df.columns:
            raise ValueError("Peak labels not created. Run create_peak_label() first.")

        self.peak_df = self.df[self.df['is_peak'] == 1].copy()

        if len(self.peak_df) == 0:
            logger.warning("No peak hours found in the data!")
            self.analysis_results = {}
            return self.analysis_results

        # Peak by various dimensions
        peak_by_hour = self.peak_df.groupby('hour_of_day').size()
        peak_by_dow = self.peak_df.groupby('day_of_week').size()
        peak_by_month = self.peak_df.groupby('month').size()
        peak_by_season = self.peak_df.groupby('season').size() if 'season' in self.peak_df.columns else None
        peak_by_weekend = self.peak_df.groupby('is_weekend').size()

        # Peak conditions
        peak_conditions = {}
        for col in ['temperature_C', 'humidity_%', 'solar_irradiation_Wm2', 'power_price_usd_per_kwh']:
            if col in self.peak_df.columns:
                peak_conditions[col] = {
                    'mean': self.peak_df[col].mean(),
                    'std': self.peak_df[col].std(),
                    'median': self.peak_df[col].median(),
                    'q25': self.peak_df[col].quantile(0.25),
                    'q75': self.peak_df[col].quantile(0.75)
                }

        self.analysis_results = {
            'peak_by_hour': peak_by_hour,
            'peak_by_dow': peak_by_dow,
            'peak_by_month': peak_by_month,
            'peak_by_season': peak_by_season,
            'peak_by_weekend': peak_by_weekend,
            'peak_conditions': peak_conditions,
            'total_peak_hours': len(self.peak_df),
            'peak_percentage': (len(self.peak_df) / len(self.df)) * 100,
            'peak_duration': self._analyze_peak_durations()
        }

        return self.analysis_results

    def _analyze_peak_durations(self) -> Dict:
        """Analyze consecutive peak hour durations"""
        peak_series = self.df['is_peak'].values
        durations = []
        current_duration = 0

        for val in peak_series:
            if val == 1:
                current_duration += 1
            else:
                if current_duration > 0:
                    durations.append(current_duration)
                    current_duration = 0
        if current_duration > 0:
            durations.append(current_duration)

        if durations:
            return {
                'max_duration': max(durations),
                'avg_duration': np.mean(durations),
                'median_duration': np.median(durations),
                'total_peak_events': len(durations)
            }
        return {}

    def plot_peak_analysis(self, save_path: str = None):
        """
        Visualize peak patterns and save as PKL
        """
        if self.analysis_results is None:
            self.analyze_peak_patterns()

        if self.analysis_results is None or len(self.analysis_results) == 0:
            logger.warning("No peak data to plot.")
            return

        fig, axes = plt.subplots(2, 3, figsize=(18, 10))
        fig.suptitle('Peak Demand Analysis', fontsize=16, fontweight='bold')

        # Peak by hour
        peak_hour = self.analysis_results['peak_by_hour'].reindex(range(24), fill_value=0)
        axes[0, 0].bar(peak_hour.index, peak_hour.values, color='red', alpha=0.7)
        axes[0, 0].set_title('Peak Hours Distribution', fontsize=12, fontweight='bold')
        axes[0, 0].set_xlabel('Hour of Day')
        axes[0, 0].set_ylabel('Peak Count')
        axes[0, 0].grid(True, alpha=0.3)
        axes[0, 0].set_xticks(range(0, 24, 2))

        # Peak by day of week
        peak_dow = self.analysis_results['peak_by_dow'].reindex(range(7), fill_value=0)
        dow_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
        axes[0, 1].bar(dow_labels, peak_dow.values, color='orange', alpha=0.7)
        axes[0, 1].set_title('Peak by Day of Week', fontsize=12, fontweight='bold')
        axes[0, 1].set_xlabel('Day')
        axes[0, 1].set_ylabel('Peak Count')
        axes[0, 1].grid(True, alpha=0.3)

        # Peak by month
        peak_month = self.analysis_results['peak_by_month'].reindex(range(1, 13), fill_value=0)
        axes[0, 2].bar(peak_month.index, peak_month.values, color='green', alpha=0.7)
        axes[0, 2].set_title('Peak by Month', fontsize=12, fontweight='bold')
        axes[0, 2].set_xlabel('Month')
        axes[0, 2].set_ylabel('Peak Count')
        axes[0, 2].grid(True, alpha=0.3)
        axes[0, 2].set_xticks(range(1, 13))

        # Temperature distribution
        if 'temperature_C' in self.df.columns:
            peak_temp = self.df[self.df['is_peak'] == 1]['temperature_C']
            non_peak_temp = self.df[self.df['is_peak'] == 0]['temperature_C']

            if len(peak_temp) > 0 and len(non_peak_temp) > 0:
                axes[1, 0].hist([non_peak_temp, peak_temp], bins=30,
                                label=['Non-Peak', 'Peak'], alpha=0.7, color=['blue', 'red'])
                axes[1, 0].set_title('Temperature: Peak vs Non-Peak', fontsize=12, fontweight='bold')
                axes[1, 0].set_xlabel('Temperature (°C)')
                axes[1, 0].set_ylabel('Frequency')
                axes[1, 0].legend()
                axes[1, 0].grid(True, alpha=0.3)

        # Peak duration analysis
        if 'peak_duration' in self.analysis_results and self.analysis_results['peak_duration']:
            durations = self.analysis_results['peak_duration']
            axes[1, 1].bar(['Max', 'Avg', 'Median'],
                          [durations['max_duration'], durations['avg_duration'], durations['median_duration']],
                          color='purple', alpha=0.7)
            axes[1, 1].set_title('Peak Duration Analysis', fontsize=12, fontweight='bold')
            axes[1, 1].set_ylabel('Hours')
            axes[1, 1].grid(True, alpha=0.3)

        # Peak probability heatmap
        if len(peak_hour) > 0 and len(peak_dow) > 0:
            heatmap_data = np.zeros((7, 24))
            for hour in range(24):
                for dow in range(7):
                    mask = (self.df['hour_of_day'] == hour) & (self.df['day_of_week'] == dow)
                    if mask.any():
                        heatmap_data[dow, hour] = self.df.loc[mask, 'is_peak'].mean() * 100

            im = axes[1, 2].imshow(heatmap_data, cmap='YlOrRd', aspect='auto', origin='lower')
            axes[1, 2].set_title('Peak Probability Heatmap (%)', fontsize=12, fontweight='bold')
            axes[1, 2].set_xlabel('Hour of Day')
            axes[1, 2].set_ylabel('Day of Week')
            axes[1, 2].set_yticks(range(7))
            axes[1, 2].set_yticklabels(dow_labels)
            axes[1, 2].set_xticks(range(0, 24, 3))
            plt.colorbar(im, ax=axes[1, 2])

        plt.tight_layout()

        if save_path:
            # Save as PKL instead of PNG
            with open(save_path.replace('.png', '.pkl'), 'wb') as f:
                pickle.dump(fig, f)
            logger.info(f"Peak analysis plot saved to {save_path.replace('.png', '.pkl')}")

        plt.show()
        return fig


# ============================================================================
# 5. OPTIMIZED MODEL TRAINING - REGRESSION (FIXED)
# ============================================================================

class OptimizedDemandForecaster:
    """
    Optimized XGBoost/LightGBM model with hyperparameter tuning for best accuracy
    """

    def __init__(self, features: List[str], target: str = 'load_demand_MW'):
        self.features = features
        self.target = target
        self.model = None
        self.scaler = RobustScaler()
        self.feature_importance = None
        self.best_params = None
        self.model_type = 'xgb'  # 'xgb' or 'lgb'
        self.is_ensemble = False
        self.models = []
        self.weights = []

    def prepare_data(self, df: pd.DataFrame, test_size: int = 168, val_size: int = 168) -> Tuple:
        """
        Prepare train/val/test split
        """
        logger.info("Preparing data with train/val/test split...")

        X = df[self.features].copy()
        y = df[self.target].copy()

        # Chronological split
        train_size = len(X) - test_size - val_size
        val_start = train_size
        test_start = train_size + val_size

        X_train = X.iloc[:train_size]
        X_val = X.iloc[val_start:test_start]
        X_test = X.iloc[test_start:]

        y_train = y.iloc[:train_size]
        y_val = y.iloc[val_start:test_start]
        y_test = y.iloc[test_start:]

        logger.info(f"Train set: {len(X_train)} samples")
        logger.info(f"Validation set: {len(X_val)} samples")
        logger.info(f"Test set: {len(X_test)} samples")

        # Scale features
        X_train_scaled = self.scaler.fit_transform(X_train)
        X_val_scaled = self.scaler.transform(X_val)
        X_test_scaled = self.scaler.transform(X_test)

        return X_train_scaled, X_val_scaled, X_test_scaled, y_train, y_val, y_test

    def optimize_hyperparameters(self, X_train, y_train, X_val, y_val, model_type: str = 'xgb'):
        """
        Bayesian-inspired hyperparameter optimization
        """
        logger.info(f"Optimizing hyperparameters for {model_type.upper()}...")
        self.model_type = model_type

        if model_type == 'xgb':
            # XGBoost hyperparameter grid
            param_grid = {
                'n_estimators': [100, 200, 300, 500, 800],
                'max_depth': [3, 4, 5, 6, 7, 8],
                'learning_rate': [0.01, 0.03, 0.05, 0.07, 0.1],
                'subsample': [0.7, 0.8, 0.9, 1.0],
                'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
                'min_child_weight': [1, 3, 5, 7],
                'reg_alpha': [0, 0.1, 0.5, 1.0],
                'reg_lambda': [0.1, 0.5, 1.0, 2.0]
            }

            # Randomized search for efficiency
            n_iter = 30
            best_score = float('inf')
            best_params = None
            best_model = None

            for i in range(n_iter):
                params = {
                    'n_estimators': np.random.choice(param_grid['n_estimators']),
                    'max_depth': np.random.choice(param_grid['max_depth']),
                    'learning_rate': np.random.choice(param_grid['learning_rate']),
                    'subsample': np.random.choice(param_grid['subsample']),
                    'colsample_bytree': np.random.choice(param_grid['colsample_bytree']),
                    'min_child_weight': np.random.choice(param_grid['min_child_weight']),
                    'reg_alpha': np.random.choice(param_grid['reg_alpha']),
                    'reg_lambda': np.random.choice(param_grid['reg_lambda'])
                }

                model = xgb.XGBRegressor(
                    **params,
                    objective='reg:squarederror',
                    random_state=42,
                    n_jobs=-1
                )

                # Fit with early stopping using eval_set
                model.fit(
                    X_train, y_train,
                    eval_set=[(X_val, y_val)],
                    verbose=False
                )

                val_pred = model.predict(X_val)
                score = mean_absolute_error(y_val, val_pred)

                if score < best_score:
                    best_score = score
                    best_params = params
                    best_model = model

                if i % 5 == 0:
                    logger.info(f"Iteration {i}, best MAE so far: {best_score:.4f}")

            self.model = best_model
            self.best_params = best_params
            logger.info(f"Best parameters: {self.best_params}")
            logger.info(f"Best validation MAE: {best_score:.4f}")

        else:  # LightGBM
            param_grid = {
                'n_estimators': [100, 200, 300, 500, 800],
                'max_depth': [3, 4, 5, 6, 7, 8, -1],
                'learning_rate': [0.01, 0.03, 0.05, 0.07, 0.1],
                'subsample': [0.7, 0.8, 0.9, 1.0],
                'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
                'min_child_samples': [5, 10, 20, 30],
                'reg_alpha': [0, 0.1, 0.5, 1.0],
                'reg_lambda': [0.1, 0.5, 1.0, 2.0]
            }

            n_iter = 30
            best_score = float('inf')
            best_params = None
            best_model = None

            for i in range(n_iter):
                params = {
                    'n_estimators': np.random.choice(param_grid['n_estimators']),
                    'max_depth': np.random.choice(param_grid['max_depth']),
                    'learning_rate': np.random.choice(param_grid['learning_rate']),
                    'subsample': np.random.choice(param_grid['subsample']),
                    'colsample_bytree': np.random.choice(param_grid['colsample_bytree']),
                    'min_child_samples': np.random.choice(param_grid['min_child_samples']),
                    'reg_alpha': np.random.choice(param_grid['reg_alpha']),
                    'reg_lambda': np.random.choice(param_grid['reg_lambda'])
                }

                model = lgb.LGBMRegressor(
                    **params,
                    objective='regression',
                    random_state=42,
                    n_jobs=-1,
                    verbose=-1
                )

                # Fit with early stopping
                model.fit(
                    X_train, y_train,
                    eval_set=[(X_val, y_val)],
                    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)]
                )

                val_pred = model.predict(X_val)
                score = mean_absolute_error(y_val, val_pred)

                if score < best_score:
                    best_score = score
                    best_params = params
                    best_model = model

                if i % 5 == 0:
                    logger.info(f"Iteration {i}, best MAE so far: {best_score:.4f}")

            self.model = best_model
            self.best_params = best_params
            logger.info(f"Best parameters: {self.best_params}")
            logger.info(f"Best validation MAE: {best_score:.4f}")

        # Feature importance
        if hasattr(self.model, 'feature_importances_'):
            self.feature_importance = pd.DataFrame({
                'feature': self.features,
                'importance': self.model.feature_importances_
            }).sort_values('importance', ascending=False)

        return self.model

    def train_ensemble(self, X_train, y_train, X_val, y_val):
        """
        Train an ensemble of models for better accuracy
        """
        logger.info("Training ensemble of models...")

        models = []
        weights = []

        # XGBoost with different seeds
        for seed in [42, 123, 456]:
            model = xgb.XGBRegressor(
                n_estimators=300,
                max_depth=6,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                min_child_weight=3,
                reg_alpha=0.1,
                reg_lambda=1.0,
                objective='reg:squarederror',
                random_state=seed,
                n_jobs=-1
            )
            model.fit(
                X_train, y_train,
                eval_set=[(X_val, y_val)],
                verbose=False
            )
            models.append(model)

            # Calculate weight based on validation performance
            val_pred = model.predict(X_val)
            weight = 1 / (mean_absolute_error(y_val, val_pred) + 1e-8)
            weights.append(weight)

        # LightGBM
        try:
            lgb_model = lgb.LGBMRegressor(
                n_estimators=300,
                max_depth=6,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                min_child_samples=20,
                reg_alpha=0.1,
                reg_lambda=1.0,
                random_state=42,
                n_jobs=-1,
                verbose=-1
            )
            lgb_model.fit(
                X_train, y_train,
                eval_set=[(X_val, y_val)],
                callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)]
            )
            models.append(lgb_model)

            val_pred = lgb_model.predict(X_val)
            weight = 1 / (mean_absolute_error(y_val, val_pred) + 1e-8)
            weights.append(weight)
        except Exception as e:
            logger.warning(f"LightGBM training failed: {e}")

        # Normalize weights
        weights = np.array(weights) / np.sum(weights)

        self.models = models
        self.weights = weights
        self.is_ensemble = True

        logger.info(f"Ensemble trained with {len(models)} models")
        logger.info(f"Ensemble weights: {weights}")

        return models

    def predict(self, X_test):
        """
        Make predictions using ensemble if available
        """
        if self.is_ensemble and len(self.models) > 0:
            predictions = np.zeros(X_test.shape[0])
            for model, weight in zip(self.models, self.weights):
                predictions += weight * model.predict(X_test)
            return predictions
        elif self.model is not None:
            return self.model.predict(X_test)
        else:
            raise ValueError("No model trained yet!")

    def evaluate(self, y_true, y_pred) -> Dict:
        """
        Calculate comprehensive regression metrics
        """
        mae = mean_absolute_error(y_true, y_pred)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
        r2 = r2_score(y_true, y_pred)

        # Additional metrics
        mpe = np.mean((y_true - y_pred) / y_true) * 100  # Mean Percentage Error
        max_error = np.max(np.abs(y_true - y_pred))

        metrics = {
            'MAE': mae,
            'RMSE': rmse,
            'MAPE': mape,
            'MPE': mpe,
            'R2': r2,
            'Max_Error': max_error
        }

        logger.info("=== Regression Metrics ===")
        for metric, value in metrics.items():
            logger.info(f"{metric}: {value:.4f}")

        return metrics

    def save_model(self, path: str):
        """
        Save model and scaler as PKL
        """
        os.makedirs(path, exist_ok=True)

        if self.is_ensemble and len(self.models) > 0:
            # Save ensemble
            model_data = {
                'models': self.models,
                'weights': self.weights,
                'is_ensemble': True
            }
            with open(f"{path}/ensemble_model.pkl", 'wb') as f:
                pickle.dump(model_data, f)
        elif self.model is not None:
            # Save single model
            with open(f"{path}/xgb_model.pkl", 'wb') as f:
                pickle.dump(self.model, f)

        # Save scaler and features
        with open(f"{path}/scaler.pkl", 'wb') as f:
            pickle.dump(self.scaler, f)

        with open(f"{path}/features.pkl", 'wb') as f:
            pickle.dump(self.features, f)

        if self.best_params:
            with open(f"{path}/best_params.pkl", 'wb') as f:
                pickle.dump(self.best_params, f)

        logger.info(f"Model saved to {path}")

    def load_model(self, path: str):
        """
        Load model and scaler from PKL
        """
        # Check if ensemble
        ensemble_path = f"{path}/ensemble_model.pkl"
        if os.path.exists(ensemble_path):
            with open(ensemble_path, 'rb') as f:
                model_data = pickle.load(f)
            self.models = model_data['models']
            self.weights = model_data['weights']
            self.is_ensemble = True
        else:
            single_path = f"{path}/xgb_model.pkl"
            if os.path.exists(single_path):
                with open(single_path, 'rb') as f:
                    self.model = pickle.load(f)
                self.is_ensemble = False

        with open(f"{path}/scaler.pkl", 'rb') as f:
            self.scaler = pickle.load(f)

        with open(f"{path}/features.pkl", 'rb') as f:
            self.features = pickle.load(f)

        logger.info(f"Model loaded from {path}")

        return self


# ============================================================================
# 6. OPTIMIZED PEAK CLASSIFIER (FIXED)
# ============================================================================

class OptimizedPeakClassifier:
    """
    Optimized classifier for peak detection with best accuracy
    """

    def __init__(self, features: List[str], target: str = 'is_peak'):
        self.features = features
        self.target = target
        self.model = None
        self.scaler = StandardScaler()
        self.feature_importance = None
        self.best_params = None

    def prepare_data(self, df: pd.DataFrame, test_size: int = 168, val_size: int = 168) -> Tuple:
        """
        Prepare data with train/val/test split
        """
        X = df[self.features].copy()
        y = df[self.target].copy()

        # Handle class imbalance check
        class_balance = y.value_counts(normalize=True)
        logger.info(f"Class balance:\n{class_balance}")

        # Chronological split
        train_size = len(X) - test_size - val_size
        val_start = train_size
        test_start = train_size + val_size

        X_train = X.iloc[:train_size]
        X_val = X.iloc[val_start:test_start]
        X_test = X.iloc[test_start:]

        y_train = y.iloc[:train_size]
        y_val = y.iloc[val_start:test_start]
        y_test = y.iloc[test_start:]

        # Scale
        X_train_scaled = self.scaler.fit_transform(X_train)
        X_val_scaled = self.scaler.transform(X_val)
        X_test_scaled = self.scaler.transform(X_test)

        return X_train_scaled, X_val_scaled, X_test_scaled, y_train, y_val, y_test

    def train_with_optimization(self, X_train, y_train, X_val, y_val):
        """
        Train classifier with hyperparameter optimization
        """
        logger.info("Training optimized peak classifier...")

        # Calculate class weight
        scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

        # Hyperparameter grid
        param_grid = {
            'n_estimators': [100, 200, 300, 500],
            'max_depth': [3, 4, 5, 6, 7],
            'learning_rate': [0.01, 0.05, 0.1, 0.15],
            'subsample': [0.7, 0.8, 0.9, 1.0],
            'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
            'min_child_weight': [1, 3, 5, 7]
        }

        # Randomized search
        n_iter = 30
        best_score = 0
        best_params = None
        best_model = None

        for i in range(n_iter):
            params = {
                'n_estimators': np.random.choice(param_grid['n_estimators']),
                'max_depth': np.random.choice(param_grid['max_depth']),
                'learning_rate': np.random.choice(param_grid['learning_rate']),
                'subsample': np.random.choice(param_grid['subsample']),
                'colsample_bytree': np.random.choice(param_grid['colsample_bytree']),
                'min_child_weight': np.random.choice(param_grid['min_child_weight'])
            }

            model = xgb.XGBClassifier(
                **params,
                scale_pos_weight=scale_pos_weight,
                objective='binary:logistic',
                random_state=42,
                n_jobs=-1,
                eval_metric='auc'
            )

            # Fit with early stopping using eval_set
            model.fit(
                X_train, y_train,
                eval_set=[(X_val, y_val)],
                verbose=False
            )

            val_pred = model.predict(X_val)
            score = f1_score(y_val, val_pred)

            if score > best_score:
                best_score = score
                best_params = params
                best_model = model

            if i % 5 == 0:
                logger.info(f"Iteration {i}, best F1 so far: {best_score:.4f}")

        self.model = best_model
        self.best_params = best_params

        logger.info(f"Best parameters: {self.best_params}")
        logger.info(f"Best validation F1: {best_score:.4f}")

        # Feature importance
        if hasattr(self.model, 'feature_importances_'):
            self.feature_importance = pd.DataFrame({
                'feature': self.features,
                'importance': self.model.feature_importances_
            }).sort_values('importance', ascending=False)

        return self.model

    def predict(self, X_test):
        """Make class predictions"""
        if self.model is None:
            raise ValueError("Model not trained yet!")
        return self.model.predict(X_test)

    def predict_proba(self, X_test):
        """Get probability scores"""
        if self.model is None:
            raise ValueError("Model not trained yet!")
        return self.model.predict_proba(X_test)

    def evaluate(self, y_true, y_pred, y_prob=None) -> Dict:
        """
        Calculate comprehensive classification metrics
        """
        precision = precision_score(y_true, y_pred)
        recall = recall_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred)

        metrics = {
            'Precision': precision,
            'Recall': recall,
            'F1-Score': f1
        }

        if y_prob is not None:
            auc = roc_auc_score(y_true, y_prob[:, 1])
            metrics['AUC-ROC'] = auc

        # Confusion matrix
        cm = confusion_matrix(y_true, y_pred)
        metrics['Confusion_Matrix'] = cm

        logger.info("=== Classification Metrics ===")
        for metric, value in metrics.items():
            if metric != 'Confusion_Matrix':
                logger.info(f"{metric}: {value:.4f}")

        logger.info(f"Confusion Matrix:\n{cm}")
        logger.info("\n" + classification_report(y_true, y_pred))

        return metrics

    def save_model(self, path: str):
        """Save classifier as PKL"""
        os.makedirs(path, exist_ok=True)

        if self.model is not None:
            with open(f"{path}/peak_classifier.pkl", 'wb') as f:
                pickle.dump(self.model, f)

        with open(f"{path}/classifier_scaler.pkl", 'wb') as f:
            pickle.dump(self.scaler, f)

        with open(f"{path}/classifier_features.pkl", 'wb') as f:
            pickle.dump(self.features, f)

        if self.best_params:
            with open(f"{path}/classifier_params.pkl", 'wb') as f:
                pickle.dump(self.best_params, f)

        logger.info(f"Classifier saved to {path}")


# ============================================================================
# 7. EVALUATION AND VISUALIZATION (PKL OUTPUTS)
# ============================================================================

class OptimizedModelEvaluator:
    """
    Comprehensive model evaluation with PKL outputs
    """

    def __init__(self, df_test: pd.DataFrame, y_true, y_pred, y_prob=None):
        self.df_test = df_test
        self.y_true = y_true
        self.y_pred = y_pred
        self.y_prob = y_prob
        self.residuals = y_true - y_pred

    def plot_predictions(self, n_points: int = 168, save_path: str = None):
        """
        Plot actual vs predicted values and save as PKL
        """
        fig, axes = plt.subplots(2, 1, figsize=(14, 10))

        plot_idx = slice(-min(n_points, len(self.y_true)), None)
        x_vals = np.arange(len(self.y_true[plot_idx]))

        # Actual vs Predicted
        axes[0].plot(x_vals, self.y_true[plot_idx], 'b-', label='Actual', linewidth=2)
        axes[0].plot(x_vals, self.y_pred[plot_idx], 'r--', label='Predicted', linewidth=2)
        axes[0].set_title('Demand Forecast: Actual vs Predicted', fontsize=14, fontweight='bold')
        axes[0].set_xlabel('Time Steps')
        axes[0].set_ylabel('Load Demand (MW)')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)

        # Residuals
        axes[1].bar(x_vals, self.residuals[plot_idx], color='green', alpha=0.6, width=1.0)
        axes[1].axhline(y=0, color='red', linestyle='--', linewidth=2)
        axes[1].set_title('Prediction Errors', fontsize=14, fontweight='bold')
        axes[1].set_xlabel('Time Steps')
        axes[1].set_ylabel('Error (MW)')
        axes[1].grid(True, alpha=0.3)

        plt.tight_layout()

        if save_path:
            # Save as PKL
            pkl_path = save_path.replace('.png', '.pkl').replace('.html', '.pkl')
            with open(pkl_path, 'wb') as f:
                pickle.dump(fig, f)
            logger.info(f"Plot saved to {pkl_path}")

        plt.show()
        return fig

    def plot_residual_analysis(self, save_path: str = None):
        """
        Analyze residuals distribution and save as PKL
        """
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))

        # Residuals distribution
        axes[0, 0].hist(self.residuals, bins=50, edgecolor='black', alpha=0.7)
        axes[0, 0].set_title('Residuals Distribution', fontsize=12, fontweight='bold')
        axes[0, 0].set_xlabel('Residual (MW)')
        axes[0, 0].set_ylabel('Frequency')
        axes[0, 0].axvline(x=0, color='red', linestyle='--', linewidth=2)

        # Q-Q plot
        stats.probplot(self.residuals, dist="norm", plot=axes[0, 1])
        axes[0, 1].set_title('Q-Q Plot', fontsize=12, fontweight='bold')
        axes[0, 1].grid(True, alpha=0.3)

        # Residuals vs Predicted
        axes[1, 0].scatter(self.y_pred, self.residuals, alpha=0.5, s=10)
        axes[1, 0].axhline(y=0, color='red', linestyle='--', linewidth=2)
        axes[1, 0].set_title('Residuals vs Predicted', fontsize=12, fontweight='bold')
        axes[1, 0].set_xlabel('Predicted (MW)')
        axes[1, 0].set_ylabel('Residual (MW)')
        axes[1, 0].grid(True, alpha=0.3)

        # Autocorrelation of residuals
        from pandas.plotting import autocorrelation_plot
        autocorrelation_plot(pd.Series(self.residuals), ax=axes[1, 1])
        axes[1, 1].set_title('Residual Autocorrelation', fontsize=12, fontweight='bold')
        axes[1, 1].set_xlim(0, 50)

        plt.tight_layout()

        if save_path:
            pkl_path = save_path.replace('.png', '.pkl')
            with open(pkl_path, 'wb') as f:
                pickle.dump(fig, f)

        plt.show()
        return fig

    def plot_feature_importance(self, feature_importance_df, top_n: int = 20, save_path: str = None):
        """
        Plot feature importance and save as PKL
        """
        top_features = feature_importance_df.head(top_n)

        fig, ax = plt.subplots(figsize=(12, 8))
        colors = plt.cm.viridis(np.linspace(0, 1, len(top_features)))

        ax.barh(range(len(top_features)), top_features['importance'].values, color=colors)
        ax.set_yticks(range(len(top_features)))
        ax.set_yticklabels(top_features['feature'].values)
        ax.set_xlabel('Importance', fontsize=12, fontweight='bold')
        ax.set_title(f'Top {top_n} Feature Importance', fontsize=14, fontweight='bold')
        ax.invert_yaxis()

        plt.tight_layout()

        if save_path:
            pkl_path = save_path.replace('.png', '.pkl')
            with open(pkl_path, 'wb') as f:
                pickle.dump(fig, f)

        plt.show()
        return fig


# ============================================================================
# 8. OPTIMIZED FORECASTING PIPELINE
# ============================================================================

class OptimizedForecastingPipeline:
    """
    End-to-end optimized forecasting pipeline with best accuracy
    """

    def __init__(self):
        self.df = None
        self.forecaster = None
        self.classifier = None
        self.feature_engineer = None
        self.feature_cols = None
        self.peak_analyzer = None

    def run_full_pipeline(self, filepath: str, test_hours: int = 168, use_ensemble: bool = True):
        """
        Execute complete optimized pipeline
        """
        logger.info("=" * 60)
        logger.info("STARTING OPTIMIZED ELECTRICITY DEMAND FORECASTING PIPELINE")
        logger.info("=" * 60)

        # 1. Load and preprocess
        loader = DataLoaderPreprocessor(filepath)
        self.df = loader.load_and_clean()
        self.df = loader.create_features(self.df)
        self.df = loader.detect_outliers(self.df)

        # 2. Advanced feature engineering
        engineer = AdvancedFeatureEngineer(self.df)
        self.df = engineer.create_lag_features()
        self.df = engineer.create_rolling_features()
        self.df = engineer.create_expanding_features()
        self.df = engineer.create_interaction_features()
        self.df = engineer.create_peak_label(threshold_percentile=90)

        # Feature correlation analysis
        corr_matrix = engineer.get_feature_correlation()

        # Remove NaN rows
        self.df = engineer.remove_nan_rows()
        self.feature_engineer = engineer

        # 3. Define features (select top correlated features)
        base_features = [
            'temperature_C', 'humidity_%', 'solar_irradiation_Wm2', 'power_price_usd_per_kwh',
            'hour_of_day', 'day_of_week', 'month', 'is_holiday', 'is_weekend',
            'hour_sin', 'hour_cos', 'day_sin', 'day_cos', 'month_sin', 'month_cos',
            'quarter', 'day_of_year'
        ]

        # Add engineered features
        lag_features = [col for col in self.df.columns if col.startswith('lag_')]
        rolling_features = [col for col in self.df.columns if 'roll_' in col]
        expanding_features = [col for col in self.df.columns if 'expanding_' in col]
        interaction_features = [col for col in self.df.columns if any(x in col for x in ['temp_', 'humidity_', 'solar_', 'price_'])]

        self.feature_cols = base_features + lag_features + rolling_features + expanding_features + interaction_features
        self.feature_cols = [f for f in self.feature_cols if f in self.df.columns]

        # Limit to top 50 features for efficiency
        if len(self.feature_cols) > 50:
            # Use correlation with target to select top features
            target_corr = self.df[self.feature_cols].corrwith(self.df['load_demand_MW']).abs()
            top_features = target_corr.sort_values(ascending=False).head(50).index.tolist()
            self.feature_cols = top_features

        logger.info(f"Total features selected: {len(self.feature_cols)}")

        # 4. Train optimized regression model
        self.forecaster = OptimizedDemandForecaster(self.feature_cols)
        X_train, X_val, X_test, y_train, y_val, y_test = self.forecaster.prepare_data(
            self.df, test_size=test_hours, val_size=test_hours//2
        )

        if use_ensemble:
            # Train ensemble for best accuracy
            self.forecaster.train_ensemble(X_train, y_train, X_val, y_val)
        else:
            # Optimize single model
            self.forecaster.optimize_hyperparameters(X_train, y_train, X_val, y_val, model_type='xgb')

        y_pred = self.forecaster.predict(X_test)

        # Evaluate regression
        reg_metrics = self.forecaster.evaluate(y_test, y_pred)

        # 5. Train optimized classifier
        self.classifier = OptimizedPeakClassifier(self.feature_cols)
        Xc_train, Xc_val, Xc_test, yc_train, yc_val, yc_test = self.classifier.prepare_data(
            self.df, test_size=test_hours, val_size=test_hours//2
        )

        self.classifier.train_with_optimization(Xc_train, yc_train, Xc_val, yc_val)
        yc_pred = self.classifier.predict(Xc_test)
        yc_prob = self.classifier.predict_proba(Xc_test)

        # Evaluate classification
        class_metrics = self.classifier.evaluate(yc_test, yc_pred, yc_prob)

        # 6. Create test dataframe
        df_test = self.df.iloc[-test_hours:].copy()
        df_test['actual_load'] = y_test
        df_test['predicted_load'] = y_pred
        df_test['peak_actual'] = yc_test
        df_test['peak_predicted'] = yc_pred
        df_test['peak_probability'] = yc_prob[:, 1]

        # 7. Peak analysis
        self.peak_analyzer = PeakAnalyzer(self.df)
        peak_patterns = self.peak_analyzer.analyze_peak_patterns()

        # 8. Generate forecast
        next_24h_forecast = self.generate_next_24h_forecast(df_test)

        # 9. Monthly aggregation
        monthly_agg = self.aggregate_monthly()

        logger.info("=" * 60)
        logger.info("PIPELINE COMPLETED SUCCESSFULLY")
        logger.info("=" * 60)

        results = {
            'regression_metrics': reg_metrics,
            'classification_metrics': class_metrics,
            'test_predictions': df_test,
            'next_24h_forecast': next_24h_forecast,
            'monthly_aggregation': monthly_agg,
            'peak_patterns': peak_patterns,
            'feature_importance': self.forecaster.feature_importance,
            'correlation_matrix': corr_matrix
        }

        return results

    def generate_next_24h_forecast(self, df_test: pd.DataFrame) -> pd.DataFrame:
        """Generate forecast for next 24 hours"""
        last_hours = df_test.tail(24).copy()

        forecast_df = pd.DataFrame({
            'hour': range(1, 25),
            'datetime': last_hours['datetime'].values if 'datetime' in last_hours.columns else None,
            'actual_load': last_hours['actual_load'].values,
            'predicted_load': last_hours['predicted_load'].values,
            'error': last_hours['actual_load'].values - last_hours['predicted_load'].values,
            'peak_risk': last_hours['peak_probability'].values * 100
        })

        forecast_df['risk_level'] = pd.cut(
            forecast_df['peak_risk'],
            bins=[0, 30, 70, 100],
            labels=['Low', 'Medium', 'High']
        )

        logger.info("\n=== Next 24 Hours Forecast ===")
        logger.info(forecast_df[['hour', 'predicted_load', 'peak_risk', 'risk_level']].to_string())

        return forecast_df

    def aggregate_monthly(self) -> pd.DataFrame:
        """Monthly aggregation for billing simulation"""
        monthly = self.df.groupby(['year', 'month']).agg({
            'load_demand_MW': ['mean', 'max', 'min', 'std', 'sum', 'median'],
            'temperature_C': 'mean',
            'is_peak': ['sum', 'mean']
        }).round(2)

        monthly.columns = ['_'.join(col).strip() for col in monthly.columns.values]
        monthly = monthly.reset_index()

        # Add month names
        month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                      'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
        monthly['month_name'] = monthly['month'].apply(lambda x: month_names[x-1])

        logger.info("\n=== Monthly Aggregation ===")
        logger.info(monthly[['year', 'month_name', 'load_demand_MW_mean', 'is_peak_sum']].head().to_string())

        return monthly

    def save_all_models(self, path: str = 'models'):
        """Save all trained models as PKL"""
        self.forecaster.save_model(f"{path}/forecaster")
        self.classifier.save_model(f"{path}/classifier")

        # Save feature engineer
        with open(f"{path}/feature_engineer.pkl", 'wb') as f:
            pickle.dump(self.feature_engineer, f)

        with open(f"{path}/all_features.pkl", 'wb') as f:
            pickle.dump(self.feature_cols, f)

        # Save pipeline configuration
        config = {
            'feature_cols': self.feature_cols,
            'n_features': len(self.feature_cols),
            'model_type': 'ensemble' if hasattr(self.forecaster, 'is_ensemble') else 'xgboost'
        }
        with open(f"{path}/pipeline_config.pkl", 'wb') as f:
            pickle.dump(config, f)

        logger.info(f"All models saved to {path}")


# ============================================================================
# 9. PRODUCTION FORECASTER (PKL BASED) - FIXED
# ============================================================================

class ProductionForecaster:
    """
    Production-ready forecasting system using PKL models
    """

    def __init__(self, model_path: str = 'models'):
        self.model_path = model_path
        self.forecaster = OptimizedDemandForecaster([])
        self.classifier = OptimizedPeakClassifier([])
        self.feature_engineer = None
        self.feature_cols = None
        self.config = None

        self.load_models()

    def load_models(self):
        """Load trained models from PKL"""
        # Load configuration
        with open(f"{self.model_path}/pipeline_config.pkl", 'rb') as f:
            self.config = pickle.load(f)

        self.feature_cols = self.config['feature_cols']

        # Load forecaster
        self.forecaster.features = self.feature_cols
        with open(f"{self.model_path}/forecaster/scaler.pkl", 'rb') as f:
            self.forecaster.scaler = pickle.load(f)

        # Check if ensemble
        ensemble_path = f"{self.model_path}/forecaster/ensemble_model.pkl"
        if os.path.exists(ensemble_path):
            with open(ensemble_path, 'rb') as f:
                model_data = pickle.load(f)
            self.forecaster.models = model_data['models']
            self.forecaster.weights = model_data['weights']
            self.forecaster.is_ensemble = True
        else:
            with open(f"{self.model_path}/forecaster/xgb_model.pkl", 'rb') as f:
                self.forecaster.model = pickle.load(f)
            self.forecaster.is_ensemble = False

        # Load classifier
        with open(f"{self.model_path}/classifier/classifier_scaler.pkl", 'rb') as f:
            self.classifier.scaler = pickle.load(f)

        with open(f"{self.model_path}/classifier/peak_classifier.pkl", 'rb') as f:
            self.classifier.model = pickle.load(f)

        self.classifier.features = self.feature_cols

        logger.info(f"Production models loaded successfully from PKL")
        logger.info(f"Model expects {len(self.feature_cols)} features")

    def prepare_new_data(self, new_data: pd.DataFrame) -> np.ndarray:
        """
        Prepare new data for prediction - ensures correct feature count
        """
        # Ensure all required features exist and in correct order
        X = pd.DataFrame(index=new_data.index)

        for col in self.feature_cols:
            if col in new_data.columns:
                X[col] = new_data[col].values
            else:
                # Create missing feature with default value (0)
                logger.warning(f"Feature '{col}' not found in input data. Using 0.")
                X[col] = 0

        # Ensure correct column order
        X = X[self.feature_cols]

        # Scale
        X_scaled = self.forecaster.scaler.transform(X)

        return X_scaled

    def predict_hourly(self, input_data: pd.DataFrame) -> pd.DataFrame:
        """
        Make hourly predictions
        """
        X = self.prepare_new_data(input_data)

        load_pred = self.forecaster.predict(X)
        peak_prob = self.classifier.predict_proba(X)[:, 1]

        results = pd.DataFrame()
        if 'datetime' in input_data.columns:
            results['datetime'] = input_data['datetime'].values

        results['predicted_load_MW'] = load_pred
        results['peak_probability'] = peak_prob
        results['peak_risk'] = (peak_prob > 0.5).astype(int)
        results['risk_level'] = pd.cut(
            peak_prob * 100,
            bins=[0, 30, 70, 100],
            labels=['Low', 'Medium', 'High']
        )

        return results


# Also fix the test section at the end of the execution code:

# ============================================================================
# 10. EXECUTION (with fixed test section)
# ============================================================================

if __name__ == "__main__":

    # File path
    FILE_PATH = "D:\electricity_project\power_system_load_cleaned.csv"

    # Create directories
    os.makedirs('output', exist_ok=True)
    os.makedirs('models', exist_ok=True)

    # Run optimized pipeline
    pipeline = OptimizedForecastingPipeline()
    results = pipeline.run_full_pipeline(FILE_PATH, test_hours=168, use_ensemble=True)

    # Visualize results (saved as PKL)
    evaluator = OptimizedModelEvaluator(
        results['test_predictions'],
        results['test_predictions']['actual_load'].values,
        results['test_predictions']['predicted_load'].values,
        np.column_stack([1-results['test_predictions']['peak_probability'].values,
                        results['test_predictions']['peak_probability'].values])
    )

    # Generate plots (saved as PKL)
    evaluator.plot_predictions(n_points=168, save_path='output/predictions.png')
    evaluator.plot_residual_analysis(save_path='output/residuals.png')

    if results['feature_importance'] is not None:
        evaluator.plot_feature_importance(results['feature_importance'], save_path='output/feature_importance.png')

    # Peak analysis plot (saved as PKL)
    pipeline.peak_analyzer.plot_peak_analysis(save_path='output/peak_analysis.png')

    # Save results as PKL and CSV
    results['test_predictions'].to_csv('output/test_predictions.csv', index=False)
    results['next_24h_forecast'].to_csv('output/next_24h_forecast.csv', index=False)
    results['monthly_aggregation'].to_csv('output/monthly_aggregation.csv', index=False)

    # Save all results as PKL
    with open('output/all_results.pkl', 'wb') as f:
        pickle.dump(results, f)

    # Save models
    pipeline.save_all_models('models')

    # Print final summary
    print("\n" + "="*70)
    print("OPTIMIZED MODEL PERFORMANCE SUMMARY")
    print("="*70)
    print("\n📊 REGRESSION METRICS:")
    for k, v in results['regression_metrics'].items():
        if k != 'Confusion_Matrix':
            print(f"   {k}: {v:.4f}")

    print("\n🎯 CLASSIFICATION METRICS:")
    for k, v in results['classification_metrics'].items():
        if k != 'Confusion_Matrix':
            print(f"   {k}: {v:.4f}")

    print("\n⏰ PEAK HOURS ANALYSIS:")
    print(f"   Total peak hours: {results['peak_patterns'].get('total_peak_hours', 0)}")
    print(f"   Peak percentage: {results['peak_patterns'].get('peak_percentage', 0):.2f}%")

    if 'peak_duration' in results['peak_patterns']:
        print(f"   Max peak duration: {results['peak_patterns']['peak_duration'].get('max_duration', 0)} hours")

    print("\n📈 TOP 5 IMPORTANT FEATURES:")
    if results['feature_importance'] is not None:
        for i, row in results['feature_importance'].head(5).iterrows():
            print(f"   {row['feature']}: {row['importance']:.4f}")

    print("\n✅ Pipeline executed successfully!")
    print("   All visualizations saved as PKL files in 'output/' directory")
    print("   Models saved as PKL files in 'models/' directory")
    print("   Results also available as CSV files in 'output/' directory")
    print("="*70)

    # Test production forecaster
    print("\n🔄 Testing Production Forecaster...")
    prod_forecaster = ProductionForecaster(model_path='models')

    # Create sample test data with only the required features
    print(f"   Model expects {len(prod_forecaster.feature_cols)} features")

    # Get the last 24 rows from the test predictions
    sample_data = results['test_predictions'].tail(24).copy()

    # Select only the features that the model was trained on
    # Some features might not be in the test_predictions DataFrame
    available_features = [col for col in prod_forecaster.feature_cols
                         if col in sample_data.columns]

    if len(available_features) < len(prod_forecaster.feature_cols):
        missing = set(prod_forecaster.feature_cols) - set(available_features)
        print(f"   Warning: Missing features in sample data: {missing}")
        print(f"   Will use default values (0) for missing features")

    # Make predictions - the prepare_new_data method will handle missing features
    prod_predictions = prod_forecaster.predict_hourly(sample_data)
    print(f"   ✅ Production forecaster ready! Made {len(prod_predictions)} predictions")
    print(f"   Sample predictions:\n{prod_predictions[['predicted_load_MW', 'peak_probability', 'risk_level']].head().to_string()}")
    print("="*70)

2026-04-11 09:31:19,234 - INFO - ============================================================
2026-04-11 09:31:19,235 - INFO - STARTING OPTIMIZED ELECTRICITY DEMAND FORECASTING PIPELINE
2026-04-11 09:31:19,236 - INFO - ============================================================
2026-04-11 09:31:19,237 - INFO - Loading data from /content/drive/MyDrive/electricity_forecast_model/power_system_load_cleaned.csv


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/electricity_forecast_model/power_system_load_cleaned.csv'